In [ ]:
import os
import re
import html
from pathlib import Path
import tkinter as tk
from tkinter import filedialog

import pandas as pd
import numpy as np
import altair as alt
import ipywidgets as widgets
from IPython.display import display, clear_output

alt.data_transformers.disable_max_rows()

fs_widget = widgets.BoundedFloatText(
    value=500,
    min=1,
    max=20000,
    step=1,
    description='fs, ??:',
    layout=widgets.Layout(width='180px')
)


In [ ]:
def extract_number(name):
    numbers = re.findall(r'\d+', str(name))
    return int(numbers[0]) if numbers else float('inf')

def generate_title_text(path):
    return Path(path).stem

def find_txt_files(folder):
    out = []
    folder = Path(folder).expanduser()
    if not folder.exists() or not folder.is_dir():
        return out

    for root, _, files in os.walk(folder):
        root_text = str(root).lower()
        if 'checkpoint' in root_text:
            continue
        for fn in files:
            fn_lower = fn.lower()
            if fn_lower.endswith('.txt') and 'checkpoint' not in fn_lower:
                out.append(str(Path(root) / fn))
    return sorted(out)

def read_txt_safely(path, delimiter='\t'):
    encodings = ['utf-8', 'utf-8-sig', 'cp1251', 'latin1', 'ascii']
    last_error = None
    for enc in encodings:
        try:
            return pd.read_csv(path, delimiter=delimiter, encoding=enc).dropna(axis=1, how='all')
        except (OSError, UnicodeDecodeError, pd.errors.ParserError, ValueError) as e:
            last_error = e
    raise ValueError(f'?? ??????? ????????? ???? {path}: {last_error}') from last_error

def is_ecg_column(col_name):
    return str(col_name).startswith('???')


In [ ]:
def get_annotation_chart(lines):
    height = max(60, 18 * len(lines))
    df = pd.DataFrame({
        'y': [i * 0.2 for i in range(len(lines))][::-1],
        'text': lines,
    })
    return alt.Chart(df).mark_text(
        align='left',
        baseline='top',
        fontSize=12,
        color='gray',
    ).encode(
        y=alt.Y('y', axis=None),
        text='text:N',
    ).properties(
        width=230,
        height=height,
    )

def get_comment_annotation():
    comments = [
        '???????????, ?????? ??????? ??? ?????',
        '5 ? ??????????????',
        '7 ? ?????? ?????',
        '9 ? ????????????',
        '13 ? ??????',
        '16 ? ???',
        '32 ? ????????????????',
    ]
    return get_annotation_chart(comments)

def get_mean_annotation(df, columns):
    text_lines = [f'{col}: {df[col].mean():.2f}' for col in columns if col in df.columns]
    if not text_lines:
        return None
    return get_annotation_chart(['???????:'] + text_lines)


In [ ]:
state = {}

def _normalize_columns(df):
    df = df.copy()
    df.columns = [str(col) for col in df.columns]
    return df

def load_file(file_path):
    df = _normalize_columns(read_txt_safely(file_path, delimiter='\t'))
    sample_rate = float(fs_widget.value)
    time = np.arange(len(df)) / sample_rate

    checkboxes = []
    hidden_B_columns = []

    for col in df.columns:
        is_hidden = col.startswith('?')
        checkboxes.append(widgets.Checkbox(value=not is_hidden, description=col))
        if is_hidden:
            hidden_B_columns.append(col)

    return {
        'path': file_path,
        'df': df,
        'time': time,
        'checkboxes': checkboxes,
        'hidden_B_columns': hidden_B_columns,
    }

def set_state(new_state):
    state.clear()
    state.update(new_state)
    return state

def get_long_df(current_state, invert_ecg=False):
    df = current_state.get('df')
    if df is None:
        return pd.DataFrame()

    selected_signals = [cb.description for cb in current_state.get('checkboxes', []) if cb.value]
    long_data = []

    for col in selected_signals:
        if col not in df.columns:
            continue
        signal = df[col].copy()
        if invert_ecg and is_ecg_column(col):
            signal = -signal
        long_data.append(pd.DataFrame({
            'Time': current_state['time'],
            'Value': signal,
            'Signal': col,
        }))

    return pd.concat(long_data, ignore_index=True) if long_data else pd.DataFrame()


In [ ]:
folder_widget = widgets.Text(
    value=str(Path.cwd()),
    description='?????:',
    layout=widgets.Layout(width='80%'),
)
choose_folder_button = widgets.Button(description='??????? ?????')
rescan_button = widgets.Button(description='???????????????')
file_dropdown = widgets.Dropdown(options=[], description='????:', disabled=True)
checkbox_box_holder = widgets.VBox([])
files_box = widgets.VBox(
    [],
    layout=widgets.Layout(
        max_height='250px',
        overflow='auto',
        border='1px solid gray',
        padding='10px',
        width='100%',
    ),
)
files_state = {'txt_files': [], 'file_checkboxes': []}
ui_state = {'suspend_dropdown_observer': False}

# ???????? ? ????????? Jupyter: ???? ?????? ??????????? ?? ??????????, ??? ??????? kernel.
def choose_folder(_):
    try:
        root = tk.Tk()
        root.withdraw()
        root.attributes('-topmost', True)
        root.update()
        folder = filedialog.askdirectory(
            initialdir=folder_widget.value or str(Path.cwd()),
            title='???????? ????? ? txt-???????',
        )
        root.destroy()
    except Exception as e:
        with plot_output:
            clear_output(wait=True)
            display(widgets.HTML(
                f"<span style='color:#b00020'>?? ??????? ??????? ????? ?????: {html.escape(str(e))}</span>"
            ))
        return

    if folder:
        folder_widget.value = folder
        on_rescan_clicked(None)

def sync_file_controls_enabled():
    has_files = bool(files_state['txt_files'])
    file_dropdown.disabled = not has_files
    for widget_name in ('invert_ecg_checkbox', 'altair_button', 'build_all_filtered_button'):
        widget = globals().get(widget_name)
        if widget is not None:
            widget.disabled = not has_files

def rescan(folder):
    txt_files = find_txt_files(folder)
    file_checkboxes = [
        widgets.Checkbox(
            value=True,
            description=generate_title_text(fp),
            layout=widgets.Layout(width='100%'),
        )
        for fp in txt_files
    ]

    files_state['txt_files'] = txt_files
    files_state['file_checkboxes'] = file_checkboxes
    files_box.children = file_checkboxes or [widgets.HTML('<i>? ????? ??? .txt ??????</i>')]

    ui_state['suspend_dropdown_observer'] = True
    try:
        file_dropdown.options = [(generate_title_text(p), p) for p in txt_files]
        file_dropdown.value = next(iter(txt_files), None)
    finally:
        ui_state['suspend_dropdown_observer'] = False

    checkbox_box_holder.children = []
    state.clear()
    sync_file_controls_enabled()
    return txt_files

rescan(folder_widget.value)


In [ ]:
plot_output = widgets.Output()

def build_chart(long_df, title, df_source, hidden_columns):
    sorted_signals = sorted(long_df['Signal'].unique(), key=extract_number)
    selection = alt.selection_point(fields=['Signal'], bind='legend')

    chart = alt.Chart(long_df).mark_line().encode(
        x='Time',
        y='Value',
        color=alt.Color('Signal:N', sort=sorted_signals),
        opacity=alt.condition(selection, alt.value(1), alt.value(0.1)),
    ).add_params(selection).properties(
        width=600,
        height=400,
        title=title,
    ).interactive()

    annotation = get_comment_annotation()
    mean_annot = get_mean_annotation(df_source, hidden_columns)
    side_panel = alt.vconcat(annotation, mean_annot) if mean_annot is not None else annotation

    return alt.hconcat(chart, side_panel).resolve_legend(color='independent').configure_view(stroke=None)

def plot_single_file(file_path):
    with plot_output:
        clear_output(wait=True)
        if not file_path or 'df' not in state:
            display(widgets.HTML('<i>??????? ???????? ????.</i>'))
            return

        long_df = get_long_df(state, invert_ecg_checkbox.value)
        if long_df.empty:
            display(widgets.HTML('<i>??? ????????? ????????.</i>'))
            return

        chart_final = build_chart(
            long_df,
            generate_title_text(file_path),
            state['df'],
            state.get('hidden_B_columns', []),
        )
        display(chart_final)


In [ ]:
def get_long_df_from_df(df_local, time_local, selected_signals, invert_ecg):
    long_data = []
    for col in selected_signals:
        if col not in df_local.columns:
            continue
        signal = df_local[col].copy()
        if invert_ecg and is_ecg_column(col):
            signal = -signal
        long_data.append(pd.DataFrame({
            'Time': time_local,
            'Value': signal,
            'Signal': col,
        }))
    return pd.concat(long_data, ignore_index=True) if long_data else pd.DataFrame()

def plot_multiple_files(selected_files, selected_signals, invert_ecg):
    with plot_output:
        clear_output(wait=True)
        if not selected_files:
            display(widgets.HTML('<i>??? ????????? ??????.</i>'))
            return
        if not selected_signals:
            display(widgets.HTML('<i>??? ????????? ????????.</i>'))
            return

        for file_path in selected_files:
            display(widgets.HTML(f'<b>????: {html.escape(generate_title_text(file_path))}</b>'))
            try:
                df_local = _normalize_columns(read_txt_safely(file_path, delimiter='\t'))
                time_local = np.arange(len(df_local)) / float(fs_widget.value)
                long_df = get_long_df_from_df(df_local, time_local, selected_signals, invert_ecg)
                if long_df.empty:
                    display(widgets.HTML('<i>??? ?????? ??? ????????? ????????.</i>'))
                    continue

                hidden_B_local = [col for col in df_local.columns if col.startswith('?')]
                chart_final = build_chart(
                    long_df,
                    generate_title_text(file_path),
                    df_local,
                    hidden_B_local,
                )
                display(chart_final)

            except (OSError, UnicodeDecodeError, pd.errors.ParserError, ValueError) as e:
                display(widgets.HTML(
                    f"<span style='color:#b00020'>?????? ??? ????????? ????? "
                    f"{html.escape(generate_title_text(file_path))}: {html.escape(str(e))}</span>"
                ))


In [ ]:
invert_ecg_checkbox = widgets.Checkbox(value=False, description='????????????? ???')
altair_button = widgets.Button(description='????????? ??????')
build_all_filtered_button = widgets.Button(description='???????? ??? ?????????')

def refresh_single_plot(path=None):
    checkbox_box_holder.children = []
    if not path:
        state.clear()
        sync_file_controls_enabled()
        return

    try:
        set_state(load_file(path))
        checkbox_box_holder.children = state['checkboxes']
    except (OSError, UnicodeDecodeError, pd.errors.ParserError, ValueError) as e:
        state.clear()
        checkbox_box_holder.children = [widgets.HTML(
            f"<span style='color:#b00020'>?? ??????? ????????? ????: {html.escape(str(e))}</span>"
        )]
        with plot_output:
            clear_output(wait=True)
            display(checkbox_box_holder.children[0])
    finally:
        sync_file_controls_enabled()

def selected_file_paths():
    return [
        fp for fp, cb in zip(files_state['txt_files'], files_state['file_checkboxes'])
        if cb.value
    ]

def selected_signal_names():
    return [cb.description for cb in state.get('checkboxes', []) if cb.value]

def on_rescan_clicked(_):
    files = rescan(folder_widget.value)
    if files:
        refresh_single_plot(file_dropdown.value)
        with plot_output:
            clear_output(wait=True)
    else:
        with plot_output:
            clear_output(wait=True)
            display(widgets.HTML('<i>? ????? ??? .txt ??????</i>'))

def on_file_change(change):
    if change['name'] != 'value' or ui_state.get('suspend_dropdown_observer'):
        return
    refresh_single_plot(change['new'])

def on_fs_change(change):
    if change['name'] != 'value' or 'df' not in state:
        return
    state['time'] = np.arange(len(state['df'])) / float(change['new'])
    if file_dropdown.value:
        plot_single_file(file_dropdown.value)

choose_folder_button.on_click(choose_folder)
rescan_button.on_click(on_rescan_clicked)
file_dropdown.observe(on_file_change, names='value')
fs_widget.observe(on_fs_change, names='value')
altair_button.on_click(lambda _: plot_single_file(file_dropdown.value))
build_all_filtered_button.on_click(lambda _: plot_multiple_files(
    selected_file_paths(),
    selected_signal_names(),
    invert_ecg_checkbox.value,
))

sync_file_controls_enabled()
if file_dropdown.value:
    refresh_single_plot(file_dropdown.value)

controls = widgets.VBox([
    widgets.HBox([folder_widget, choose_folder_button, rescan_button]),
    fs_widget,
    file_dropdown,
    checkbox_box_holder,
    invert_ecg_checkbox,
    altair_button,
    widgets.HTML('<b>????? ??? ??????-??????:</b>'),
    files_box,
    build_all_filtered_button,
])

display(widgets.VBox([controls, plot_output]))
